In [4]:
#离线阶段
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain.chat_models import init_chat_model
from app.core.config import settings
# 在线阶段
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from app.core.config import settings
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()

# =============1.准备文档================
with open('../resources/评估.md', 'r', encoding='utf-8') as f:
    doc = f.read()

# =============2.切分文档================
# 切分依据，这里是按照三级标题
headers_to_split_on = [
    ("#", "h1"),
    ("##", "h2"),
    ("###", "h3"),
]

# 创建切分器
markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on,
    strip_headers=False # 在正文中是否去除标题，False，不去除
)

# 切分文档, 返回的是 list[Document]
docs = markdown_splitter.split_text(doc)

for i, doc in enumerate(docs, start=1):
    doc.id = f'doc_{i}'

# 1.初始化模型
model = init_chat_model(
    "deepseek-v4-flash",
    api_key=settings.llm.api_key,
    extra_body={'thinking': {'type': 'disabled'}}
)
# 1.1.初始化向量模型
ollama_embeddings = OllamaEmbeddings(
    model="qwen3-embedding:0.6b",  # 性价比高的模型
    dimensions=1024  # 可选：减少维度以节省存储
)
# 1.3.初始化向量数据库
vector_store = InMemoryVectorStore(embedding=ollama_embeddings)
# 1.4保存文档到向量数据库
ids = vector_store.add_documents(docs)

# 2.定义工具
@tool
def retrieve_docs(query: str):
    """检索知识库，获取知识片段"""
    # 检索知识片段, lisi[Document]
    # k=2，意味着每次只从向量库捞 2 个片段
    docs = vector_store.similarity_search(query, k=2)
    # 拼接返回，不需要我们组织提示词，tool结果会直接给模型
    return "\n\n".join(doc.page_content for doc in docs)


# 3.初始化Agent，绑定模型和工具
agent = create_agent(
    model=model,
    tools=[retrieve_docs],
    system_prompt='''
    你是一个教资考试专家，你调用工具检索文档回答用户问题，文档中没有的不要编造，直接说不知道。
    回答直接引用文档中原文，不要自己发挥！
    '''
)
# 4.调用Agent
query = "论语中教育的目的是什么？"
response = agent.invoke({'messages': [HumanMessage(content=query)]})

for message in response['messages']:
    message.pretty_print()

================================ Human Message =================================

论语中教育的目的是什么？
================================== Ai Message ==================================
Tool Calls:
  retrieve_docs (call_00_SkdFfMyDFLIpJ7q0wRdx0628)
 Call ID: call_00_SkdFfMyDFLIpJ7q0wRdx0628
  Args:
    query: 论语 教育目的
================================= Tool Message =================================
Name: retrieve_docs

### （三）孔子及《论语》主要思想  
**教育作用**：庶、富、教；性相近，习相远。  
**教育对象**：“有教无类”，教育民主思想。  
**教育目的**：以完善人格为教育的首要目的，培养士和君子。  
**教学内容**：文、行、忠、义。  
**教学过程**：学、思、习、行。  
**教学原则**：因材施教、启发诱导、学思结合、谦虚笃实。  
**教师观**：其身正，不令而行；其身不正，虽令不从。

### （四）孟子主要思想  
**人性论**：人性本善，人先天具有仁、义、礼、智四个“善端”。  
**教育作用**：发扬善端，培养道德完人，得天下英才而教育之。  
**教学原则**：循序渐进，专心有恒。
================================== Ai Message ==================================

根据文档，《论语》中（孔子）的教育目的是：**以完善人格为教育的首要目的，培养士和君子。**
